In [0]:
from pyspark.sql.functions import *

In [0]:
customers_path = "/Volumes/dbacademy/get_started_de/myfiles/customers.csv"
products_path = "/Volumes/dbacademy/get_started_de/myfiles/products.csv"
orders_path = "/Volumes/dbacademy/get_started_de/myfiles/orders_day1.csv"

In [0]:
customers_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(customers_path)
    .select("*", "_metadata")
)

In [0]:
customers_bronze = (
    customers_df
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name"))
    .withColumn("source_path", col("_metadata.file_path"))
    .drop("_metadata")
)

In [0]:
customers_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_customers")

In [0]:
products_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(products_path)
    .select("*", "_metadata")
)

In [0]:
products_bronze = (
    products_df
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name"))
    .withColumn("source_path", col("_metadata.file_path"))
    .drop("_metadata")
)


In [0]:
products_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_products")

In [0]:
orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(orders_path)
    .select("*", "_metadata")
)

orders_bronze = (
    orders_df
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", col("_metadata.file_name"))
    .withColumn("source_path", col("_metadata.file_path"))
    .drop("_metadata")
)

orders_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_orders")

In [0]:
orders_bronze.filter(
    col("customer_id").isNull()
).show()

In [0]:
orders_bronze.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()